# Dzien 2 / Krok 2: Feature Engineering i model bazowy

**Input:** `checkpoints/01_merged.parquet`
**Output:** `checkpoints/02_train.parquet`, `checkpoints/02_test.parquet`

Budujemy cechy, trenujemy prosty model bazowy i oceniamy go.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_parquet("checkpoints/01_merged.parquet")
print(f"Zaladowano: {df.shape}")
print(f"Target rate: {df['is_bad_experience'].mean():.3f}")


## Trend i sezonowosc wolumenu zamowien

Zanim zbudujemy cechy, zobaczmy jak wyglada wolumen zamowien w czasie.
Z tego szeregu wyciagniemy dwa sygnaly, ktore wejda do modelu jako cechy.

Intuicja:
- **Trend**: platforma rosnie - wiecej zamowien = wieksze obciazenie logistyki = potencjalnie wiecej opoznien
- **Sezonowosc**: niektore miesiace sa historycznie wyzsze (np. Black Friday) - zmienia profil ryzyka

Sciaga: `notebooks/building_blocks_and_examples/seasonality.ipynb`


In [ ]:
# TODO: Oblicz miesięczny wolumen zamowien i narysuj wykres
# Wskazowka: df.set_index("order_purchase_timestamp").resample("ME")["order_id"].count()
df_monthly = ___

# TODO: narysuj go.Scatter z wolumenem per miesiac


### Dekompozycja: trend + sezonowosc + reszta

Uzywamy `seasonal_decompose` z biblioteki statsmodels.

1. Trend - srednia ruchoma z oknem 12 miesiecy
2. Sezonowosc - srednie odchylenie od trendu per miesiac roku
3. Reszta - to czego trend i sezon nie wyjasniaja


In [ ]:
# TODO: Wykonaj dekompozycje sezonowa df_monthly
# Wskazowka: seasonal_decompose(df_monthly, model="additive", period=12)
result = ___

# TODO: Narysuj 4 komponenty (oryginal, trend, seasonal, resid) na 4 subplotach


### Mapowanie komponentow na zamowienia

In [ ]:
# TODO: Zbuduj lookup DataFrame: Period(rok-miesiac) -> (volume_trend, volume_seasonal)
# Wskazowka: pd.DataFrame({"volume_trend": result.trend.interpolate(method="linear", limit_direction="both"),
#                          "volume_seasonal": result.seasonal})
#            index = df_monthly.index.to_period("M")
df_decomp = ___

# TODO: Zmapuj na kazde zamowienie przez rok-miesiac
# Wskazowka: df["_period"] = df["order_purchase_timestamp"].dt.to_period("M")
#            df["volume_trend"] = df["_period"].map(df_decomp["volume_trend"])
df["_period"]         = ___
df["volume_trend"]    = ___
df["volume_seasonal"] = ___
df = df.drop(columns=["_period"])

print(f"volume_trend NaN:    {df['volume_trend'].isna().sum()}")
print(f"volume_seasonal NaN: {df['volume_seasonal'].isna().sum()}")


## Budowa cech: build_features()

In [1]:
# Docelowo chcemy miec 1 funkcje ktora tworzy wszystkie cechy, w fazie trial & error warto tworzyć je w inline w komórkach

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Buduje cechy do modelu.
    """
    feat = df.copy()

    # TODO: estimated_delivery_days = roznica miedzy estimated_delivery_date a purchase_timestamp w dniach
    # Wskazowka: (feat["col_A"] - feat["col_B"]).dt.days.clip(0, 120)
    feat["estimated_delivery_days"] = ___

    # TODO: approval_time_hours = czas od zakupu do zatwierdzenia platnosci w godzinach
    # Wolna akceptacja moze sygnalizowac problemy platnicze lub system antyfrauda.
    # Wskazowka: .dt.total_seconds().div(3600).clip(0, 48).fillna(24)
    feat["approval_time_hours"] = ___

    # TODO: Cechy czasowe z order_purchase_timestamp
    # purchase_hour (0-23), purchase_dow (0=pn, 6=nd), purchase_month (1-12), is_weekend (0/1)
    feat["purchase_hour"]  = ___
    feat["purchase_dow"]   = ___
    feat["purchase_month"] = ___
    feat["is_weekend"]     = ___

    # TODO: price_log = np.log1p(price),  freight_ratio = freight_value / price.clip(lower=1)
    feat["price_log"]     = ___
    feat["freight_ratio"] = ___

    # TODO: Platnosci - fillna(0) dla installments, is_installment gdy > 1
    feat["payment_installments"] = ___
    feat["is_installment"]       = ___

    # TODO: LabelEncoder dla kolumn kategorycznych -> nowa kolumna col + "_enc"
    # Kolumny: payment_type, product_category_name_english, customer_state, seller_state
    for col in ["payment_type", "product_category_name_english",
                "customer_state", "seller_state"]:
        le = LabelEncoder()
        feat[col + "_enc"] = ___

    # TODO: Uzupelnij braki: product_weight_g -> mediana, product_photos_qty -> 1
    feat["product_weight_g"]   = ___
    feat["product_photos_qty"] = ___

    return feat

df_feat = build_features(df)
print(f"Cechy zbudowane: {df_feat.shape}")


In [ ]:
FEATURE_COLS = [
    "estimated_delivery_days", "approval_time_hours",
    "purchase_hour", "purchase_dow", "purchase_month", "is_weekend",
    "volume_trend", "volume_seasonal",
    "price_log", "freight_ratio",
    "items_count", "payment_installments", "is_installment",
    "product_weight_g", "product_photos_qty",
    "payment_type_enc", "product_category_name_english_enc",
    "customer_state_enc", "seller_state_enc",
]

print(f"Liczba cech: {len(FEATURE_COLS)}")
missing_total = df_feat[FEATURE_COLS].isna().sum().sum()
print("Braki: OK" if missing_total == 0 else f"UWAGA: {missing_total} brakow - sprawdz build_features()")


## Train/test split

Uzywamy `stratify=y`, zeby zachowac proporcje klas w obu zbiorach.

Bez `stratify`: przy losowym podziale na niezbalansowanych danych (~10% klasy 1)
proporcja klasy pozytywnej moze sie roznic miedzy train i testem.
Z `stratify=y`: sklearn gwarantuje identyczny procent klasy 1 w obu zbiorach.


In [ ]:
X = df_feat[FEATURE_COLS].copy()
y = df_feat["is_bad_experience"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Bad rate train: {y_train.mean():.3f}  |  Bad rate test: {y_test.mean():.3f}")
print(f"scale_pos_weight: {scale_pos_weight:.1f}")


## Model bazowy: XGBoost

### Niezbalansowane klasy i scale_pos_weight

~10% zlych recenzji. Bez korekcji model przewiduje "dobra recenzja" dla wszystkiego.

```
scale_pos_weight = liczba_negatywnych / liczba_pozytywnych  ≈ 9
```

Efekt: model "widzi" kazdy przyklad zlej recenzji 9x ciezej.


In [ ]:
import xgboost as xgb

model_baseline = xgb.XGBClassifier(
    objective        = "binary:logistic",
    n_estimators     = 300,
    max_depth        = 5,
    learning_rate    = 0.05,
    scale_pos_weight = scale_pos_weight,
    eval_metric      = "auc",
    random_state     = 42,
    n_jobs           = -1,
    # todo: dodaj własny hiperparametr z dokumantacji XGBClassifier i zaobserwuj jego wpływ
)
model_baseline.fit(X_train, y_train,
                   eval_set=[(X_test, y_test)], verbose=False)
print("Trening zakonczony.")


In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)

def evaluate(model, X_test, y_test, threshold=0.5, name=""):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)
    auc     = roc_auc_score(y_test, y_proba)
    report  = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    cm      = confusion_matrix(y_test, y_pred)

    cm_T = cm.T  # wiersze = Predicted, kolumny = Actual
    labels = ["0 (dobra)", "1 (zla)"]

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Macierz pomylek", "Krzywa ROC"])
    fig.add_trace(go.Heatmap(z=cm_T, text=cm_T, texttemplate="%{text}",
                             x=labels, y=labels,
                             colorscale="Blues", showscale=False), row=1, col=1)
    fig.update_xaxes(title_text="Actual", row=1, col=1)
    fig.update_yaxes(title_text="Predicted", row=1, col=1)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"AUC={auc:.3f}",
                             line=dict(color="steelblue", width=2)), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                             line=dict(dash="dash", color="gray"), showlegend=False), row=1, col=2)
    fig.update_xaxes(title_text="False Positive Rate (FPR)", row=1, col=2)
    fig.update_yaxes(title_text="True Positive Rate (Recall)", row=1, col=2)
    fig.update_layout(title=f"{name}  |  AUC={auc:.4f}", height=420)
    fig.show()

    p1 = report.get("1", {})
    print(f"  Precision(1): {p1.get('precision',0):.3f}")
    print(f"  Recall(1):    {p1.get('recall',0):.3f}")
    print(f"  F1(1):        {p1.get('f1-score',0):.3f}")
    return {"auc": auc, "precision_1": p1.get("precision",0),
            "recall_1": p1.get("recall",0), "f1_1": p1.get("f1-score",0)}

metrics_baseline = evaluate(model_baseline, X_test, y_test, name="XGBoost Baseline")


In [ ]:
# Threshold sweep - jak prog decyzji wplywa na Precision i Recall
thresholds = np.arange(0.1, 0.9, 0.02)
y_proba = model_baseline.predict_proba(X_test)[:, 1]
sweep_rows = []
for t in thresholds:
    pred = (y_proba >= t).astype(int)
    tp = int(((y_test==1) & (pred==1)).sum())
    fp = int(((y_test==0) & (pred==1)).sum())
    fn = int(((y_test==1) & (pred==0)).sum())
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0
    sweep_rows.append({"threshold": t, "precision": p, "recall": r,
                       "fn_count": fn, "fp_count": fp})

df_thresh = pd.DataFrame(sweep_rows)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Precision i Recall vs prog",
                                    "Liczba FN i FP vs prog"])
fig.add_trace(go.Scatter(x=df_thresh.threshold, y=df_thresh.precision,
                         name="Precision", line=dict(color="steelblue")), row=1, col=1)
fig.add_trace(go.Scatter(x=df_thresh.threshold, y=df_thresh.recall,
                         name="Recall", line=dict(color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=df_thresh.threshold, y=df_thresh.fn_count,
                         name="False Negatives", line=dict(color="crimson")), row=1, col=2)
fig.add_trace(go.Scatter(x=df_thresh.threshold, y=df_thresh.fp_count,
                         name="False Positives", line=dict(color="steelblue")), row=1, col=2)
fig.update_layout(height=400, title="Jak prog decyzji wplywa na metryki")
fig.show()

TARGET_RECALL = 0.70
optimal = df_thresh[df_thresh.recall >= TARGET_RECALL].iloc[-1]
print(f"Przy recall >= {TARGET_RECALL}: prog={optimal.threshold:.2f}, "
      f"precision={optimal.precision:.3f}, FN={optimal.fn_count:.0f}")


## Checkpoint: zapisz zbior cech

In [ ]:
df_train = X_train.copy()
df_train["is_bad_experience"] = y_train.values
df_test  = X_test.copy()
df_test["is_bad_experience"]  = y_test.values

df_train.to_parquet("checkpoints/02_train.parquet", index=False)
df_test.to_parquet( "checkpoints/02_test.parquet",  index=False)

print(f"  checkpoints/02_train.parquet  {df_train.shape}")
print(f"  checkpoints/02_test.parquet   {df_test.shape}")


## Feature Importance: SHAP vs XGBoost

**XGBoost gain** - ile razy cecha byla uzywana w podzialach, wazone przez poprawe straty.

**SHAP** - dla kazdego przykladu liczy wklad kazdej cechy w konkretna predykcje.
Mean |SHAP| = sredni bezwzgledny wplyw na wynik modelu.

Sciaga: `notebooks/building_blocks_and_examples/eda_feature_engineering.ipynb` (sekcja Feature Importance)


In [ ]:
import shap
import matplotlib.pyplot as plt

# TODO: Stworz TreeExplainer i oblicz shap_values dla X_test
explainer   = ___
shap_values = ___

# TODO: Oblicz srednie |SHAP| per cecha, posortuj rosnacc
df_shap_imp = ___

xgb_scores = model_baseline.get_booster().get_score(importance_type="gain")
df_xgb_imp = pd.Series(xgb_scores).reindex(FEATURE_COLS).fillna(0)
df_xgb_imp = (df_xgb_imp / df_xgb_imp.max()).reindex(df_shap_imp.index)

# TODO: Narysuj SHAP vs XGBoost gain na 2 subplotach (go.Bar, orientation="h")


In [ ]:
# TODO: Beeswarm plot - kazdy punkt = jedno zamowienie
# Wskazowka: shap.summary_plot(shap_values, X_test, plot_type="dot", max_display=15)


## PoC gotowe

Masz dzialajacy model z feature importance.
- Ktore cechy sa najwazniejsze? Czy to ma sens biznesowo?
- Co mozna by poprawic w Feature Engineering?
